<a href="https://colab.research.google.com/github/Jirtus-sanasam/MLP-Diabetes/blob/main/DiabeteXGB1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              classification_report, confusion_matrix)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('/content/diabetes_data2.csv')

# Replace 0s with NaN
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols] = df[cols].replace(0, np.nan)

# KNN Imputation
knn_imputer = KNNImputer(n_neighbors=5)
df[cols] = knn_imputer.fit_transform(df[cols])

# Cap Outliers with IQR
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return df

for col in df.columns[:-1]:
    df = cap_outliers(df, col)

In [3]:
# Interaction Features
df['Glucose_BMI']           = df['Glucose'] * df['BMI']
df['Glucose_Insulin']       = df['Glucose'] * df['Insulin']
df['BMI_Age']               = df['BMI'] * df['Age']
df['Insulin_SkinThickness'] = df['Insulin'] * df['SkinThickness']

# Ratio Features
df['Glucose_Insulin_Ratio'] = df['Glucose'] / (df['Insulin'] + 1e-6)
df['BMI_Age_Ratio']         = df['BMI'] / (df['Age'] + 1e-6)

# Polynomial Features
df['Glucose_sq'] = df['Glucose'] ** 2
df['BMI_sq']     = df['BMI'] ** 2
df['Age_sq']     = df['Age'] ** 2

# Age Binning
df['Age_Group'] = pd.cut(df['Age'], bins=[0,30,45,60,100],
                          labels=['Young','Middle','Senior','Elderly'])
df = pd.get_dummies(df, columns=['Age_Group'], drop_first=True, dtype=int)

# BMI Category
df['BMI_Category'] = pd.cut(df['BMI'], bins=[0,18.5,24.9,29.9,100],
                              labels=['Underweight','Normal','Overweight','Obese'])
df = pd.get_dummies(df, columns=['BMI_Category'], drop_first=True, dtype=int)

# Glucose Category
df['Glucose_Level'] = pd.cut(df['Glucose'], bins=[0,99,125,300],
                               labels=['Normal','Prediabetes','Diabetes'])
df = pd.get_dummies(df, columns=['Glucose_Level'], drop_first=True, dtype=int)

# HOMA-IR
df['HOMA_IR'] = (df['Glucose'] * df['Insulin']) / 405

# Pregnancy Risk Flag
df['High_Pregnancies'] = (df['Pregnancies'] > 3).astype(int)

# Log Transforms
df['Log_Insulin']                  = np.log1p(df['Insulin'])
df['Log_DiabetesPedigreeFunction'] = np.log1p(df['DiabetesPedigreeFunction'])

# Composite Risk Score
df['Risk_Score'] = (df['Glucose']/100) + (df['BMI']/25) + \
                   (df['Age']/50) + df['DiabetesPedigreeFunction']

# Bool → int safety check
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

print(df.shape)

(768, 31)


In [4]:
selected_features = ['Glucose', 'Glucose_BMI', 'Glucose_Insulin', 'BMI_Age',
                     'Insulin_SkinThickness', 'BMI_Age_Ratio', 'Glucose_sq',
                     'BMI_sq', 'HOMA_IR', 'Risk_Score']

X = df[selected_features]
Y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Train shape: (614, 10)
Test shape : (154, 10)


In [5]:
# XGBoost doesn't require scaling, but it helps with SMOTE distance calculations
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [6]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", dict(pd.Series(y_train).value_counts()))
print("After SMOTE :", dict(pd.Series(y_train_sm).value_counts()))

Before SMOTE: {0: np.int64(400), 1: np.int64(214)}
After SMOTE : {0: np.int64(400), 1: np.int64(400)}


In [7]:
def build_xgb():
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_alpha=0.1,       # L1 regularization
        reg_lambda=1.0,      # L2 regularization
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    )

In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_accuracy, cv_auc, cv_f1 = [], [], []

X_full = scaler.fit_transform(X)
Y_arr  = Y.values

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, Y_arr)):
    print(f"\n--- Fold {fold+1} ---")

    X_tr, X_val = X_full[train_idx], X_full[val_idx]
    y_tr, y_val = Y_arr[train_idx],  Y_arr[val_idx]

    # Apply SMOTE inside each fold to prevent leakage
    smote = SMOTE(random_state=42)
    X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)

    # Train XGBoost with early stopping
    model = build_xgb()
    model.fit(
        X_tr_sm, y_tr_sm,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    y_pred_prob = model.predict_proba(X_val)[:, 1]
    y_pred      = (y_pred_prob >= 0.5).astype(int)

    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_pred_prob)
    f1  = f1_score(y_val, y_pred)

    cv_accuracy.append(acc)
    cv_auc.append(auc)
    cv_f1.append(f1)

    print(f"  Accuracy: {acc:.4f} | AUC: {auc:.4f} | F1: {f1:.4f}")

print("\n========== CV Summary ==========")
print(f"Accuracy : {np.mean(cv_accuracy):.4f} ± {np.std(cv_accuracy):.4f}")
print(f"AUC-ROC  : {np.mean(cv_auc):.4f} ± {np.std(cv_auc):.4f}")
print(f"F1 Score : {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")


--- Fold 1 ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:35:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Accuracy: 0.7468 | AUC: 0.8352 | F1: 0.6486

--- Fold 2 ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:35:23] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Accuracy: 0.7922 | AUC: 0.8546 | F1: 0.7037

--- Fold 3 ---


...

  Accuracy: 0.7857 | AUC: 0.8463 | F1: 0.7179

--- Fold 4 ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:35:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Accuracy: 0.7320 | AUC: 0.8074 | F1: 0.6496

--- Fold 5 ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:35:26] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Accuracy: 0.6536 | AUC: 0.7481 | F1: 0.5827

========== CV Summary ==========
Accuracy : 0.7421 ± 0.0497
AUC-ROC  : 0.8183 ± 0.0386
F1 Score : 0.6605 ± 0.0479


In [9]:
final_xgb = build_xgb()
final_xgb.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False
)

y_test_prob = final_xgb.predict_proba(X_test_scaled)[:, 1]
y_test_pred = (y_test_prob >= 0.5).astype(int)

print("\n=== Final Test Set Results ===")
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"AUC-ROC  : {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"F1 Score : {f1_score(y_test, y_test_pred):.4f}")
print("\n", classification_report(y_test, y_test_pred))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:35:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== Final Test Set Results ===
Accuracy : 0.7273
AUC-ROC  : 0.8156
F1 Score : 0.6250

               precision    recall  f1-score   support

           0       0.80      0.77      0.79       100
           1       0.60      0.65      0.62        54

    accuracy                           0.73       154
   macro avg       0.70      0.71      0.71       154
weighted avg       0.73      0.73      0.73       154

